# MCH–MFT Track Classification — Training & Testing

Regular XGBoost to classify (MCH, MFT candidate) pairs as one of the match labels


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils
import seaborn as sns
import onnxruntime as ort


from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax
from scipy.stats import ks_2samp
from sklearn.inspection import permutation_importance
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.utils.class_weight import compute_class_weight


pd.set_option('display.max_columns', None)

In [ ]:
importlib.reload(Utils)


## 1. Load, Format, Engineer Data

In [ ]:
df = Utils.get_dataframe("OO-LHC25i4_FIXED.root", folder_name="DF_*")
# df = Utils.get_dataframe("PbPbLHC26b13_FIXED.root", folder_name="DF_*")


In [ ]:
df.describe()

In [ ]:
# np.seterr(all='raise')
df = Utils.process_dataframe(df, makedummies=False)

# ── Map raw MatchLabel to grouped categories ──────────────────────────────────
# Use the dictionary in Utils.py and a stable class ordering.
CATEGORY_NAMES = ["True", "Wrong", "Decay", "Fake"]
raw_to_category = {}
for category_name in CATEGORY_NAMES:
    if category_name not in Utils.MATCH_LABEL_GROUPS:
        raise KeyError(f"Expected category '{category_name}' in Utils.MATCH_LABEL_GROUPS")
    for raw_value in Utils.MATCH_LABEL_GROUPS[category_name]:
        raw_to_category[raw_value] = CATEGORY_NAMES.index(category_name)

# Create category column
if 'MatchLabel' not in df.columns:
    raise KeyError('Expected MatchLabel column in dataframe')
df['MatchLabel_Category'] = df['MatchLabel'].map(raw_to_category).fillna(-1).astype(int)
missing_mask = df['MatchLabel_Category'] == -1
if missing_mask.any():
    missing_values = sorted(df.loc[missing_mask, 'MatchLabel'].unique())
    raise ValueError(f"Found raw MatchLabel values with no category mapping: {missing_values}")

print('Label mapping order:', CATEGORY_NAMES)
print('Label distribution:')
print(df['MatchLabel_Category'].value_counts().sort_index())
print('\nLabel mapping: 0=True, 1=Wrong, 2=Decay, 3=Fake')

TARGET = "MatchLabel_Category"

GROUP  = "mchID"

FEATURES = [
    f for f in df.columns.tolist()
    if f not in Utils.NON_TRAINING_FEATURES + ['MatchLabel_Category']
]

In [ ]:
METRICS = ["prob_True", "prob_Wrong", "prob_Decay", "prob_Fake"]

In [ ]:
# ### Downsample the DataFrame to X% of the original size while maintaining the distribution of 'mchID'
# # 1. Get unique group IDs
# unique_ids = df["mchID"].unique()

# # 2. Sample X% of IDs
# n_sample = int(0.6 * len(unique_ids))
# sampled_ids = np.random.choice(unique_ids, n_sample, replace=False)

# # 3. Filter rows belonging to those IDs
# df = df[df["mchID"].isin(sampled_ids)]

In [ ]:
FEATURES

## 2. Sanity Checks

In [ ]:
n_mch_tracks = df["mchID"].nunique()
candidates_per_track = df.groupby("mchID").size()
label_dist = df[TARGET].value_counts().sort_index()
label_names = ["True", "Wrong", "Decay", "Fake"]

print(f"MCH tracks:          {n_mch_tracks:,}")
print(f"Total pairs:         {len(df):,}")
print(f"\nLabel distribution (multiclass):")
for lbl in range(4):
    count = (df[TARGET] == lbl).sum()
    pct = 100 * count / len(df)
    print(f"  {label_names[lbl]:<6} (class {lbl}): {count:>7,} ({pct:>5.1f}%)")
print(f"\nCandidates per track: min={candidates_per_track.min()}, "
      f"max={candidates_per_track.max()}, "
      f"mean={candidates_per_track.mean():.2f}")
print('Data loaded and preprocessed. Ready for training.')

## 4. Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test)

In [ ]:
groups   = df[GROUP].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42) # shuffle the data and split into train/test sets while ensuring all candidates from the same mchID are in the same set
train_idx, test_idx = next(splitter.split(df, groups=groups))

df_train = df.iloc[train_idx]
df_test  = df.iloc[test_idx]

print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)")
print(f"\nTrain label distribution:")
label_names = ["True", "Wrong", "Decay", "Fake"]
for lbl in range(4):
    count = (df_train[TARGET] == lbl).sum()
    pct = 100 * count / len(df_train)
    print(f"  {label_names[lbl]:<6} (class {lbl}): {count:>7,} ({pct:>5.1f}%)")
print(f"\nTest label distribution:")
for lbl in range(4):
    count = (df_test[TARGET] == lbl).sum()
    pct = 100 * count / len(df_test)
    print(f"  {label_names[lbl]:<6} (class {lbl}): {count:>7,} ({pct:>5.1f}%)")
print("Data split into training and testing sets.")

## 5. Train XGBoost tree

In [ ]:
# Compute class weights for multi-class imbalance
classes = np.arange(4)
class_weights = compute_class_weight('balanced', classes=classes, y=df_train[TARGET])
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"Class weights for balanced training: {class_weight_dict}")

model = xgb.XGBClassifier(
    objective="multi:softmax",  # Multi-class classification
    num_class=4,  # 4 classes: True, Wrong, Decay, Fake
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",  # Multi-class log loss
    early_stopping_rounds=5,
    random_state=42,
    n_jobs=5, # Back to assigning manually due to crashes for now
    tree_method="hist",
)

model.fit(
    df_train[FEATURES], df_train[TARGET],
    eval_set=[(df_train[FEATURES], df_train[TARGET]), (df_test[FEATURES], df_test[TARGET])],
    verbose=True,
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Training curve ────────────────────────────────────────────────────────────
results   = model.evals_result()
train_loss = results["validation_0"]["mlogloss"]
test_loss  = results["validation_1"]["mlogloss"]
iterations = range(1, len(train_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — full training curve
ax = axes[0]
ax.plot(iterations, train_loss, lw=2, color="steelblue", label="Train Loss")
ax.plot(iterations, test_loss,  lw=2, color="tomato",    label="Test Loss")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("Multi-class Log Loss")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

# Right — zoomed to last 20%
zoom_start = int(0.8 * len(train_loss))
ax = axes[1]
ax.plot(list(iterations)[zoom_start:], train_loss[zoom_start:],
        lw=2, color="steelblue", label="Train Loss")
ax.plot(list(iterations)[zoom_start:], test_loss[zoom_start:],
        lw=2, color="tomato",    label="Test Loss")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("Multi-class Log Loss")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_loss[-1] - test_loss[-1])
best_gap  = abs(train_loss[model.best_iteration - 1] -
                test_loss[model.best_iteration - 1])
print(f"Train/test loss gap at best iteration: {best_gap:.6f}")
print(f"Train/test loss gap at final iteration: {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.05 else 'No significant overfitting detected'}")

## 6. Feature Importance

In [ ]:
importances = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(100, 100))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances")
plt.tight_layout()
plt.show()
print(importances)

In [ ]:
importances[[x in Utils.DESIGNED_FEATURES for x in importances.index]]

# Read in trained model for evaluation

In [ ]:
# sess = ort.InferenceSession("model_CLASSIFIER_OO.onnx")
# input_name = sess.get_inputs()[0].name

In [ ]:
# # df_test["score"] =
# print(sess.run(
#     None,
#     {input_name: df_test[FEATURES].to_numpy(dtype=np.float32)}
# )[0].ravel()[0])

## 6a. Multi-class Evaluation (Confusion Matrix & Classification Metrics)

In [ ]:
df_test = df_test.copy()

# Get class predictions and probabilities for all 4 classes
class_predictions = model.predict(df_test[FEATURES])
class_probabilities = model.predict_proba(df_test[FEATURES])

df_test["predicted_class"] = class_predictions

# Store probabilities for each class
label_names = ["True", "Wrong", "Decay", "Fake"]
for class_idx, class_name in enumerate(label_names):
    df_test[f"prob_{class_name}"] = class_probabilities[:, class_idx]

# Main score: probability of True class (class 0)
df_test["score"] = class_probabilities[:, 0]

print("Prediction columns added to df_test:")
print(f"  - predicted_class: predicted class (0-3)")
print(f"  - prob_True, prob_Wrong, prob_Decay, prob_Fake: class probabilities")
print(f"  - score: probability of True class (used for ranking)")

In [ ]:
df_test.head()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# df_leader_thresholded = df_test.loc[df_test.groupby("mchID")["prob_True"].idxmax()].reset_index(drop=True)
# df_leader_thresholded = df_leader_thresholded[df_leader_thresholded["prob_True"] > 0.5]
df_with_cut = df_test[(df_test[['prob_True',	'prob_Wrong',	'prob_Decay',	'prob_Fake']] > 0.9).any(axis=1)]


df_cm = df_with_cut

# Get predictions on test set
y_pred = model.predict(df_cm[FEATURES])
y_true = df_cm[TARGET]

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, normalize='true') # , normalize='none' , normalize='true'
label_names = ["True", "Wrong", "Decay", "Fake"]

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='f', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names,
            ax=ax, cbar_kws={'label': 'Density'})
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()

# Print classification metrics
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_names))
print(f"\nOverall Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# Per-class metrics
print("\nPer-class Performance:")
for i, label in enumerate(label_names):
    tn = cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]
    tp = cm[i, i]
    fp = cm[:, i].sum() - cm[i, i]
    fn = cm[i].sum() - cm[i, i]
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"  {label:<8} - Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

## 7. Permutation feature importance

In [ ]:
def compute_permutation_importance(
    model, 
    X_val, 
    y_val, 
    features,
    n_repeats=10,
    random_state=42,
    n_jobs=5
):
    """
    Compute permutation importance and return as a sorted Series.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation feature matrix
    y_val : validation labels
    features : list of feature names
    n_repeats : number of times to permute (default 10)
    random_state : for reproducibility
    n_jobs : parallel jobs (-1 = all cores)
    
    Returns
    -------
    perm_df : DataFrame with importance, std, and percentile
    """
    
    perm_result = permutation_importance(
        model, 
        X_val, 
        y_val,
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=n_jobs
    )
    
    perm_df = pd.DataFrame({
        'feature': features,
        'importance': perm_result.importances_mean,
        'std': perm_result.importances_std,
    }).sort_values('importance', ascending=True)
    
    perm_df['importance_percentile'] = perm_df['importance'].rank(pct=True)
    
    return perm_df


def plot_permutation_importance(
    perm_df,
    figsize=(10, 10),
    title="Permutation Importance (Validation Set)",
    show_threshold=None,
    threshold_label=None
):
    """
    Plot permutation importance with error bars.
    
    Parameters
    ----------
    perm_df : DataFrame from compute_permutation_importance
    figsize : figure size
    title : plot title
    show_threshold : optional importance threshold line to plot
    threshold_label : label for threshold line
    
    Returns
    -------
    fig, ax : matplotlib figure and axis
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    y_pos = np.arange(len(perm_df))
    ax.barh(
        y_pos, 
        perm_df['importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(perm_df['feature'])
    ax.set_xlabel("Permutation Importance (error bars = ±1 std)")
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
    
    if show_threshold is not None:
        ax.axvline(
            show_threshold, 
            color='red', 
            linestyle='--', 
            linewidth=2,
            label=threshold_label or f'Threshold = {show_threshold:.4f}'
        )
        ax.legend()
    
    plt.tight_layout()
    
    return fig, ax


def compare_importances(
    model,
    X_val,
    y_val,
    features,
    figsize=(14, 10),
    n_repeats=10,
    random_state=42,
):
    """
    Compute and plot both native XGBoost importance and permutation importance side-by-side.
    
    Parameters
    ----------
    model : fitted XGBoost model
    X_val : validation features
    y_val : validation labels
    features : list of feature names
    figsize : figure size
    n_repeats : number of permutation repeats
    random_state : for reproducibility
    
    Returns
    -------
    perm_df : DataFrame with permutation importance
    fig, axes : matplotlib figure and axes
    """
    
    # Native importance
    native_imp = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=True)
    
    # Permutation importance
    perm_df = compute_permutation_importance(
        model, X_val, y_val, features,
        n_repeats=n_repeats,
        random_state=random_state
    )
    
    # Create side-by-side plots
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Native importance (left)
    axes[0].barh(range(len(native_imp)), native_imp.values, alpha=0.8, edgecolor='black', linewidth=0.5)
    axes[0].set_yticks(range(len(native_imp)))
    axes[0].set_yticklabels(native_imp.index)
    axes[0].set_xlabel("Feature Importance (gain)")
    axes[0].set_title("XGBoost Native Importance")
    axes[0].grid(axis='x', alpha=0.3)
    
    # Permutation importance (right)
    y_pos = np.arange(len(perm_df))
    axes[1].barh(
        y_pos,
        perm_df['importance'],
        xerr=perm_df['std'],
        error_kw={'ecolor': 'gray', 'alpha': 0.5, 'capsize': 3},
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(perm_df['feature'])
    axes[1].set_xlabel("Permutation Importance (error bars = ±1 std)")
    axes[1].set_title("Permutation Importance (Validation)")
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    
    return perm_df, fig, axes


# Example usage:
"""
# Assuming you have: model, X_val, y_val, FEATURES

# Option 1: Just permutation importance
perm_df = compute_permutation_importance(model, X_val, y_val, FEATURES)
fig, ax = plot_permutation_importance(perm_df)
plt.show()

# Option 2: Compare side-by-side with native importance
perm_df, fig, axes = compare_importances(model, X_val, y_val, FEATURES)
plt.show()

# Option 3: Apply filtering
perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
fig, ax = plot_permutation_importance(
    perm_df_filtered,
    title="Permutation Importance (Top 60%)",
    show_threshold=perm_df[perm_df['importance_percentile'] == 0.4]['importance'].values[0],
    threshold_label="40th percentile cutoff"
)
plt.show()

print(perm_df[['feature', 'importance', 'std', 'importance_percentile']].to_string())
"""


# should this be uuh train or test?
# perm_df = compute_permutation_importance(model, X_test, y_test, FEATURES)
# fig, ax = plot_permutation_importance(perm_df)
# plt.show()

# perm_df, fig, axes = compare_importances(model, X_test, y_test, FEATURES)
# plt.show()


# TODO: add a plot of the difference between ranking boost & permutation importance - see discrepancy and say smth

In [ ]:
# # perm_df = compute_permutation_importance(model, X_test, y_test, FEATURES) unneeded as it is already calculated in the compare_importances function
# perm_df_filtered = perm_df[perm_df['importance_percentile'] > 0.4]
# fig, ax = plot_permutation_importance(
#     perm_df_filtered,
#     title="Permutation Importance (Top 60%)",
#     show_threshold=perm_df[perm_df['importance_percentile'] >= 0.4]['importance'].values[0],
#     threshold_label="40th percentile cutoff"
# )
# plt.show()

# print(perm_df[['feature', 'importance', 'std', 'importance_percentile']].to_string())

In [ ]:


# # Get native importance as DataFrame
# native_df = pd.DataFrame({
#     'feature': FEATURES,
#     'native_importance': model.feature_importances_
# })

# # Prepare permutation DataFrame
# perm_df_renamed = perm_df[['feature', 'importance']].rename(columns={'importance': 'perm_importance'})

# # Merge the two
# importance_comparison = pd.merge(native_df, perm_df_renamed, on='feature')

# # Normalize each importance metric to [0,1] by dividing by its maximum
# importance_comparison['native_importance_norm'] = importance_comparison['native_importance'] / importance_comparison['native_importance'].max()
# importance_comparison['perm_importance_norm'] = importance_comparison['perm_importance'] / importance_comparison['perm_importance'].max()

# # Compute difference (normalized permutation - normalized native)
# importance_comparison['difference'] = importance_comparison['perm_importance_norm'] - importance_comparison['native_importance_norm']

# # Sort by absolute difference for better visualization
# importance_comparison = importance_comparison.sort_values('difference', key=abs, ascending=False)

# # Plot
# fig, ax = plt.subplots(figsize=(12, 8))
# bars = ax.barh(importance_comparison['feature'], importance_comparison['difference'],
#                color=['red' if x < 0 else 'blue' for x in importance_comparison['difference']],
#                alpha=0.7)

# ax.set_xlabel('Normalized Difference (Permutation - Native Importance)')
# ax.set_title('Normalized Difference between Permutation and Native Feature Importance')
# ax.grid(axis='x', alpha=0.3)

# # Add value labels on bars
# for bar, diff in zip(bars, importance_comparison['difference']):
#     width = bar.get_width()
#     ax.text(width + (0.01 if width >= 0 else -0.01), bar.get_y() + bar.get_height()/2,
#             f'{diff:.3f}', ha='left' if width >= 0 else 'right', va='center', fontsize=8)

# plt.tight_layout()
# plt.show()

# # Also print the comparison table
# print("Normalized Feature Importance Comparison:")
# print(importance_comparison[['feature', 'native_importance_norm', 'perm_importance_norm', 'difference']].round(4))

## 8. Correlation diversion

In [ ]:
#TODO: move all these functions to utils for maintainability

def find_correlation_groups(
    X,
    correlation_threshold=0.75,
    method='complete',
    drop_constant=True,
    verbose=True
):
    """
    Find groups of correlated features using hierarchical clustering.
    
    Parameters
    ----------
    X : feature matrix (DataFrame or array)
    correlation_threshold : features with |correlation| >= this are grouped
    method : linkage method ('complete', 'average', 'single')
    drop_constant : remove zero-variance features before clustering
    verbose : print diagnostics
    
    Returns
    -------
    groups : dict mapping group_id -> list of feature names
    linkage_matrix : for dendrogram plotting
    features : list of features actually used (after filtering)
    corr_matrix : correlation matrix
    dropped_features : list of features removed due to data quality issues
    """
    
    if isinstance(X, pd.DataFrame):
        features = X.columns.tolist()
        X_array = X.values
    else:
        features = [f'feature_{i}' for i in range(X.shape[1])]
        X_array = X
    
    # Check for problematic features
    dropped_features = []
    valid_idx = []
    valid_features = []
    
    for i, feat in enumerate(features):
        col = X_array[:, i]
        
        # Check for constant (zero variance)
        if np.var(col) == 0:
            if drop_constant:
                dropped_features.append((feat, 'zero variance'))
                continue
        
        # Check for NaN
        if np.any(np.isnan(col)):
            dropped_features.append((feat, 'contains NaN'))
            continue
        
        # Check for inf
        if np.any(np.isinf(col)):
            dropped_features.append((feat, 'contains inf'))
            continue
        
        valid_idx.append(i)
        valid_features.append(feat)
    
    if verbose and dropped_features:
        print(f"Dropped {len(dropped_features)} features due to data quality:")
        for feat, reason in dropped_features:
            print(f"  - {feat}: {reason}")
    
    # Subset X to valid features
    X_clean = X_array[:, valid_idx]
    
    # Compute correlation and distance
    corr_matrix = np.corrcoef(X_clean.T)
    
    # Check for NaN in correlation matrix (still might happen with edge cases)
    if np.any(np.isnan(corr_matrix)):
        nan_pairs = np.where(np.isnan(corr_matrix))
        if verbose:
            print(f"Warning: {len(nan_pairs[0])} NaN values in correlation matrix")
        # Replace NaN with 0 (uncorrelated)
        corr_matrix = np.nan_to_num(corr_matrix, nan=0.0)
    
    # Convert correlation to distance (1 - |correlation|)
    distance_matrix = 1 - np.abs(corr_matrix)
    
    # Hierarchical clustering
    linkage_matrix = linkage(
        distance_matrix[np.triu_indices_from(distance_matrix, k=1)],
        method=method
    )
    
    # Cut dendrogram at threshold
    distance_threshold = 1 - correlation_threshold
    cluster_labels = fcluster(linkage_matrix, distance_threshold, criterion='distance')
    
    # Build groups
    groups = {}
    for label, feature in zip(cluster_labels, valid_features):
        if label not in groups:
            groups[label] = []
        groups[label].append(feature)
    
    if verbose:
        print(f"Clustered {len(valid_features)} features into {len(groups)} groups (ρ threshold = {correlation_threshold})")
    
    return groups, linkage_matrix, valid_features, corr_matrix, dropped_features
 
 
def select_from_correlation_groups(
    groups,
    perm_df,
    corr_matrix,
    features,
    method='permutation_importance'
):
    """
    From each correlation group, select the most informative feature.
    
    Parameters
    ----------
    groups : dict from find_correlation_groups
    perm_df : DataFrame from compute_permutation_importance
    corr_matrix : correlation matrix
    features : list of all feature names
    method : 'permutation_importance' (best) or 'ks_stat' if you have KS
    
    Returns
    -------
    selected_features : list of features to keep
    group_info : DataFrame showing which features were dropped from each group
    """
    
    selected = []
    group_info = []
    
    for group_id, group_features in groups.items():
        if len(group_features) == 1:
            # No correlation, keep it
            selected.append(group_features[0])
            continue
        
        # Get importance for features in this group
        group_importance = perm_df[perm_df['feature'].isin(group_features)].copy()
        
        if len(group_importance) == 0:
            # Features not in perm_df (shouldn't happen), keep first
            selected.append(group_features[0])
            continue
        
        # Select best by permutation importance
        best_idx = group_importance['importance'].idxmax()
        best_feature = group_importance.loc[best_idx, 'feature']
        selected.append(best_feature)
        
        # Log what was dropped
        dropped = group_importance[group_importance['feature'] != best_feature]['feature'].tolist()
        for drop_feat in dropped:
            max_corr_in_group = np.abs(
                corr_matrix[
                    features.index(drop_feat),
                    [features.index(f) for f in group_features if f != drop_feat]
                ]
            ).max()
            
            group_info.append({
                'kept_feature': best_feature,
                'dropped_feature': drop_feat,
                'kept_importance': group_importance[group_importance['feature'] == best_feature]['importance'].values[0],
                'dropped_importance': group_importance[group_importance['feature'] == drop_feat]['importance'].values[0],
                'max_correlation_in_group': max_corr_in_group,
                'group_size': len(group_features)
            })
    
    group_info_df = pd.DataFrame(group_info)
    
    return selected, group_info_df
 
 
def plot_correlation_heatmap(
    corr_matrix,
    features,
    selected_features=None,
    figsize=(12, 10),
    threshold=0.75
):
    """
    Plot correlation heatmap with selected features highlighted.
    
    Parameters
    ----------
    corr_matrix : correlation matrix
    features : list of feature names
    selected_features : features to highlight (optional)
    figsize : figure size
    threshold : show threshold line on colorbar
    """
    
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        np.abs(corr_matrix),  # Use absolute correlation
        xticklabels=features,
        yticklabels=features,
        cmap='RdYlBu_r',
        center=0.5,
        vmin=0,
        vmax=1,
        cbar_kws={'label': '|Correlation|'},
        ax=ax,
        square=True,
        linewidths=0.5
    )
    
    ax.set_title(f'Feature Correlation Matrix (|ρ| threshold = {threshold})')
    plt.tight_layout()
    
    return fig, ax
 
 
# ============================================================================
# EXAMPLE WORKFLOW
# ============================================================================
 

# # 1. Find correlated groups (with data quality checks)
# groups, linkage_mat, features, corr_matrix, dropped_features = find_correlation_groups(
#     X_train.loc[:, (X_train != 0).any(axis=0)],  # Drop all-zero columns
#     correlation_threshold=0.75
# )
 
# print(f"Found {len(groups)} groups")
# for gid, feats in groups.items():
#     if len(feats) > 1:
#         print(f"  Group {gid}: {feats}")
 
# # 2. Select best from each group using permutation importance
# selected_features, group_info = select_from_correlation_groups(
#     groups, 
#     perm_df,  # from your permutation importance pipeline
#     corr_matrix,
#     features
# )
 
# print(f"\nSelected {len(selected_features)} features from {len(features)} original")
# print("\nDropped features:")
# print(group_info[['kept_feature', 'dropped_feature', 'kept_importance', 'dropped_importance', 'max_correlation_in_group']])
 
# # 3. Visualize
# fig, ax = plot_correlation_heatmap(corr_matrix, features, selected_features, threshold=0.75)
# plt.show()
 
# # 4. Train on reduced feature set
# X_filtered = X_train[selected_features]


## 9. Group-Level Evaluation

In [ ]:
df_test.head()

In [ ]:
# g = xgb.to_graphviz(
#     model,
#     num_trees=10,
#     graph_attr={"dpi": "300", "size": "20,10"}
# )

# g.render("xgb_tree", format="png", cleanup=True)

## 10. All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=True)

## 11. Leading Match Metric Distribution Analysis

In [ ]:
df_leader = df_test.loc[df_test.groupby("mchID")["prob_True"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=True, per=0.0)

## 12. Metric score sweep

In [ ]:
importlib.reload(Utils)

for entry in METRICS:
    Utils.sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0, n_steps=100)

## Return of the softmax

## 13. Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.6
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
df_leader = df_leader[df_leader["score"] >= threshold].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True, per=0.01)

## 14. Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
mch_cols = ["PhiMCH", "TanlMCH", "InvQPtMCH", "PtMCH"] # MCH cols we can actually use for these metrics as they require the underlying group structure to be preserved
common_cols= mch_cols + ['PDCA', 'Rabs', 'MFTMult']
#TODO: confirm these preserve grouping. Add non mch group preserving structures... like possibly MFTMult if it remains defined based on the best chi2 tracks around a mch track - correspond t
for entry in common_cols:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.6, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.05, trim_high=0.005, Nsigma=1.0)

## 15. Delta Investigation

In [ ]:
df_group_eval = df_test.sort_values(["mchID", "score"], ascending=[True, False])

In [ ]:
df_group_eval["rank"] = df_group_eval.groupby("mchID").cumcount()

top1 = df_group_eval[df_group_eval["rank"] == 0][["mchID", "score"]].rename(columns={"score": "max_score"})
top2 = df_group_eval[df_group_eval["rank"] == 1][["mchID", "score"]].rename(columns={"score": "second_max_score"})

In [ ]:
group_stats = top1.merge(top2, on="mchID", how="left")
group_stats["second_max_score"] = group_stats["second_max_score"].fillna(0.0)

group_stats["delta_1_2"] = group_stats["max_score"] - group_stats["second_max_score"]

In [ ]:
agg_stats = df_group_eval.groupby("mchID")["score"].agg(["mean", "std"]).reset_index()
agg_stats = agg_stats.rename(columns={"mean": "mean_score", "std": "std_score"})

In [ ]:
labels = df_group_eval.groupby("mchID")[TARGET].apply(lambda x: (x == 0).any()).reset_index()
labels = labels.rename(columns={TARGET: "has_true_match"})

In [ ]:
group_stats = group_stats.merge(agg_stats, on="mchID")
group_stats = group_stats.merge(labels, on="mchID")

In [ ]:
true_groups = group_stats[group_stats["has_true_match"] == 1]
false_groups = group_stats[group_stats["has_true_match"] == 0]

In [ ]:

plt.hist(true_groups["max_score"], bins=50, alpha=0.5, label="True-match groups")
plt.hist(false_groups["max_score"], bins=50, alpha=0.5, label="No-match groups")
plt.xlabel("Max score per group")
plt.ylabel("Count")
plt.legend()
plt.title("Group-level max score distribution")
plt.show()

In [ ]:
plt.hist(true_groups["delta_1_2"], bins=50, alpha=0.5, label="True-match groups")
plt.hist(false_groups["delta_1_2"], bins=50, alpha=0.5, label="No-match groups")
plt.xlabel("Score gap: top1 - top2")
plt.ylabel("Count")
plt.legend()
plt.title("Group-level score separation (delta)")
plt.show()

In [ ]:
plt.figure(figsize=(7,6))

sns.kdeplot(
    data=true_groups,
    x="max_score",
    y="mean_score",
    cmap="Reds",
    fill=True,
    alpha=0.5,
    label="True-match"
)

# sns.kdeplot(
#     data=false_groups,
#     x="max_score",
#     y="mean_score",
#     cmap="Blues",
#     fill=True,
#     alpha=0.5,
#     label="No-match"
# )

plt.title("Density: True vs No-match groups")
plt.legend()
plt.show()

In [ ]:
plt.scatter(false_groups["mean_score"], false_groups["max_score"], alpha=0.3, label="No-match groups")
plt.xlabel("Mean score")
plt.ylabel("Max score")
plt.legend()
plt.title("Group score structure")
plt.show()


## 15. Feature Selection

In [ ]:
# ── Thresholds — adjust these ─────────────────────────────────────────────────
KS_THRESHOLD         = 0.1
IMPORTANCE_THRESHOLD = 0.01

# ── Compute KS statistic for each feature ────────────────────────────────────
signal     = df[df["IsSignal"] == 1]
background = df[df["IsSignal"] == 0]

ks_stats = {
    feature: ks_2samp(signal[feature].dropna(),
                      background[feature].dropna()).statistic
    for feature in FEATURES
}

importance = dict(zip(FEATURES, model.feature_importances_))

selection_df = pd.DataFrame({
    "ks":         ks_stats,
    "importance": importance,
}).sort_values("ks", ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

passes = selection_df[
    (selection_df["ks"]         >= KS_THRESHOLD) &
    (selection_df["importance"] >= IMPORTANCE_THRESHOLD)
]
fails = selection_df[
    (selection_df["ks"]         <  KS_THRESHOLD) |
    (selection_df["importance"] <  IMPORTANCE_THRESHOLD)
]

ax.scatter(fails["ks"],   fails["importance"],
           alpha=0.6, color="tomato",    s=60, label="Rejected")
ax.scatter(passes["ks"],  passes["importance"],
           alpha=0.8, color="steelblue", s=60, label="Selected")

for feature, row in selection_df.iterrows():
    ax.annotate(feature, (row["ks"], row["importance"]),
                fontsize=8, textcoords="offset points", xytext=(5, 3))

ax.axvline(KS_THRESHOLD,         color="black", linestyle="--", lw=1.2,
           label=f"KS threshold ({KS_THRESHOLD})")
ax.axhline(IMPORTANCE_THRESHOLD, color="grey",  linestyle="--", lw=1.2,
           label=f"Importance threshold ({IMPORTANCE_THRESHOLD})")

ax.set_xlabel("KS statistic (univariate separability)")
ax.set_ylabel("Feature importance (model usage)")
ax.set_title("Feature selection overview")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Print selected features ───────────────────────────────────────────────────
selected = passes.index.tolist()
dropped  = fails.index.tolist()

print(f"Selected ({len(selected)}) — copy-paste ready:")
print("FEATURES = [")
for f in selected:
    print(f"    \"{f}\",")
print("]")

print(f"\nDropped ({len(dropped)}):")
print(dropped)
print(selection_df.round(4))

## SHAP Addition

In [ ]:
import shap
from matplotlib import cm


def compute_shap_importance_scalable(
    model,
    X_data,
    feature_names=None,
    sample_size=None,
    top_k=25,
    viz_modes=['bar_top_k', 'beeswarm_top_k', 'dependence_grid'],
    **kwargs
):
    """
    Compute SHAP values and generate scalable visualizations for many features.
    
    Args:
        model: Trained XGBoost model
        X_data: Feature matrix (np.array or pd.DataFrame)
        feature_names: List of feature names
        sample_size: Max samples for explainer (e.g., 5000 for speed)
        top_k: Number of top features to visualize in detail (default 25)
        viz_modes: List of visualizations to generate
                   Options: 'bar_all', 'bar_top_k', 'beeswarm_top_k', 
                           'dependence_grid', 'heatmap_top_k'
        **kwargs: Extra args for compatibility
    
    Returns:
        dict with SHAP values, explainer, importance dataframe, and vis objects
    """
    
    # Extract feature names and convert to array
    if feature_names is None:
        if isinstance(X_data, pd.DataFrame):
            feature_names = X_data.columns.tolist()
            X_array = X_data.values
        else:
            feature_names = [f"Feat_{i}" for i in range(X_data.shape[1])]
            X_array = X_data
    else:
        X_array = X_data.values if isinstance(X_data, pd.DataFrame) else X_data
    
    n_features = len(feature_names)
    print(f"\n{'='*60}")
    print(f"Computing SHAP for {n_features} features, {len(X_array)} samples")
    print(f"{'='*60}\n")
    
    # Subsample for explainer if needed
    if sample_size is not None and len(X_array) > sample_size:
        idx = np.random.choice(len(X_array), size=sample_size, replace=False)
        X_sample = X_array[idx]
        print(f"Using {sample_size} samples for explainer (subsampled from {len(X_array)})")
    else:
        X_sample = X_array
        sample_size = len(X_array)
    
    # Create explainer and compute SHAP values
    print("Computing TreeSHAP values...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    # Handle multi-class
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    
    # Compute importance metrics
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Mean |SHAP|': mean_abs_shap,
        'Rank': np.arange(1, n_features + 1)
    }).sort_values('Mean |SHAP|', ascending=False).reset_index(drop=True)
    importance_df['Rank'] = np.arange(1, len(importance_df) + 1)
    
    print("\n=== Top 15 Features by Mean |SHAP| ===")
    print(importance_df.head(15)[['Rank', 'Feature', 'Mean |SHAP|']].to_string(index=False))
    print(f"\n... and {n_features - 15} more features")
    
    # Visualizations
    print(f"\nGenerating visualizations: {viz_modes}")
    figs = {}
    
    if 'bar_all' in viz_modes:
        figs['bar_all'] = _plot_bar_all_features(importance_df)
    
    if 'bar_top_k' in viz_modes:
        figs['bar_top_k'] = _plot_bar_top_k(importance_df, top_k=top_k)
    
    if 'beeswarm_top_k' in viz_modes:
        figs['beeswarm_top_k'] = _plot_beeswarm_top_k(
            shap_values, X_sample, importance_df, feature_names, top_k=top_k
        )
    
    if 'dependence_grid' in viz_modes:
        figs['dependence_grid'] = _plot_dependence_grid(
            shap_values, X_sample, importance_df, feature_names, top_k=min(top_k, 12)
        )
    
    if 'heatmap_top_k' in viz_modes:
        figs['heatmap_top_k'] = _plot_heatmap_top_k(
            shap_values, importance_df, feature_names, top_k=top_k, n_samples_plot=100
        )
    
    return {
        'shap_values': shap_values,
        'explainer': explainer,
        'X_sample': X_sample,
        'importance_df': importance_df,
        'feature_names': feature_names,
        'mean_abs_shap': mean_abs_shap,
        'figs': figs
    }


def _plot_bar_all_features(importance_df, figsize=(14, 20)):
    """Bar plot of all features (tall, scrollable)."""
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = cm.RdYlGn_r(np.linspace(0.2, 0.8, len(importance_df)))
    ax.barh(importance_df['Feature'], importance_df['Mean |SHAP|'], color=colors)
    ax.set_xlabel('Mean Absolute SHAP Value', fontsize=11)
    ax.set_title(f'SHAP Importance: All {len(importance_df)} Features', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig('shap_bar_all_features.png', dpi=100, bbox_inches='tight')
    print("✓ Saved: shap_bar_all_features.png")
    return fig


def _plot_bar_top_k(importance_df, top_k=25, figsize=(10, 8)):
    """Bar plot of top-k features."""
    df_top = importance_df.head(top_k)
    
    fig, ax = plt.subplots(figsize=figsize)
    colors = cm.viridis(np.linspace(0, 1, len(df_top)))
    ax.barh(df_top['Feature'], df_top['Mean |SHAP|'], color=colors)
    ax.set_xlabel('Mean Absolute SHAP Value', fontsize=11)
    ax.set_title(f'Top {top_k} Features by SHAP Importance', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(f'shap_bar_top_{top_k}.png', dpi=100, bbox_inches='tight')
    print(f"✓ Saved: shap_bar_top_{top_k}.png")
    return fig


def _plot_beeswarm_top_k(shap_values, X_sample, importance_df, feature_names, top_k=25):
    """
    Beeswarm plot for top-k features.
    Each point = one sample. Color = feature value. Y-axis = SHAP value.
    Shows distribution and direction of SHAP contributions.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['Feature']]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    y_pos = 0
    y_labels = []
    y_ticks = []
    
    for rank, (feature, idx) in enumerate(zip(df_top['Feature'], top_indices)):
        shap_vals = shap_values[:, idx]
        feature_vals = X_sample[:, idx]
        
        # Normalize feature values to [0, 1] for coloring
        fv_norm = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-8)
        colors = cm.coolwarm(fv_norm)
        
        # Add jitter to avoid overplotting
        y_jitter = np.random.normal(y_pos, 0.04, size=len(shap_vals))
        ax.scatter(shap_vals, y_jitter, c=colors, s=30, alpha=0.6, edgecolor='none')
        
        y_labels.append(f"#{rank+1}: {feature}")
        y_ticks.append(y_pos)
        y_pos += 1
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel('SHAP Value (← negative impact | positive impact →)', fontsize=11)
    ax.set_title(f'SHAP Beeswarm: Top {top_k} Features\n(Color: feature value, cold→hot)', 
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    fig.tight_layout()
    fig.savefig(f'shap_beeswarm_top_{top_k}.png', dpi=100, bbox_inches='tight')
    print(f"✓ Saved: shap_beeswarm_top_{top_k}.png")
    return fig


def _plot_dependence_grid(shap_values, X_sample, importance_df, feature_names, top_k=12):
    """
    Grid of dependence plots: SHAP value vs feature value for top-k features.
    Each subplot shows one feature.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['Feature']]
    
    n_cols = 3
    n_rows = (len(df_top) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = axes.flatten()
    
    for idx, (feature, feat_idx) in enumerate(zip(df_top['Feature'], top_indices)):
        ax = axes[idx]
        
        shap_vals = shap_values[:, feat_idx]
        feature_vals = X_sample[:, feat_idx]
        
        # Color by feature value
        sc = ax.scatter(feature_vals, shap_vals, c=feature_vals, cmap='coolwarm', 
                       s=40, alpha=0.6, edgecolor='none')
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel(feature, fontsize=10)
        ax.set_ylabel('SHAP Value', fontsize=10)
        ax.set_title(f'{feature}', fontsize=11, fontweight='bold')
        plt.colorbar(sc, ax=ax, label='Feature Value')
        ax.grid(alpha=0.3)
    
    # Hide unused subplots
    for idx in range(len(df_top), len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle(f'SHAP Dependence Plots: Top {top_k} Features', 
                fontsize=14, fontweight='bold', y=1.00)
    fig.tight_layout()
    fig.savefig(f'shap_dependence_top_{top_k}.png', dpi=100, bbox_inches='tight')
    print(f"✓ Saved: shap_dependence_top_{top_k}.png")
    return fig


def _plot_heatmap_top_k(shap_values, importance_df, feature_names, top_k=25, n_samples_plot=100):
    """
    Heatmap: samples × top-k features, with SHAP values as cell colors.
    Good for spotting patterns and interactions.
    """
    df_top = importance_df.head(top_k)
    top_indices = [feature_names.index(f) for f in df_top['Feature']]
    
    # Subsample rows if too many
    if len(shap_values) > n_samples_plot:
        idx = np.random.choice(len(shap_values), size=n_samples_plot, replace=False)
        shap_sub = shap_values[idx]
    else:
        shap_sub = shap_values
    
    data_heat = shap_sub[:, top_indices]
    
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(data_heat, cmap='RdBu_r', aspect='auto', vmin=-np.abs(data_heat).max(), 
                   vmax=np.abs(data_heat).max())
    
    ax.set_yticks(np.arange(len(shap_sub)))
    ax.set_xticks(np.arange(len(df_top)))
    ax.set_xticklabels(df_top['Feature'], rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels([f"Sample {i}" for i in range(len(shap_sub))], fontsize=8)
    
    cbar = plt.colorbar(im, ax=ax, label='SHAP Value')
    ax.set_title(f'SHAP Heatmap: Top {top_k} Features × {len(shap_sub)} Samples\n(Blue=negative, Red=positive)', 
                fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(f'shap_heatmap_top_{top_k}.png', dpi=100, bbox_inches='tight')
    print(f"✓ Saved: shap_heatmap_top_{top_k}.png")
    return fig

In [ ]:
# results = compute_shap_importance_scalable(
#     model=model,
#     X_data=df_test[FEATURES],
#     feature_names=FEATURES,
#     sample_size=5000,
#     top_k=25,
#     viz_modes=['bar_top_k', 'beeswarm_top_k', 'dependence_grid', 'heatmap_top_k']
# )

# # Access results
# print(results['importance_df'].head(20))

# # Individual feature analysis
# fig = results['figs']['beeswarm_top_k']

## Uniqueness checks

In [ ]:
grouped = df.groupby("mchID")

# number of unique values per group, per column
nunique_per_group = grouped.nunique()

# columns where ALL groups have exactly 1 unique value
constant_cols = nunique_per_group.eq(1).all(axis=0)

# get column names
constant_cols = constant_cols[constant_cols].index.tolist()
print(constant_cols)

In [ ]:
# def plot_nunique_histograms(df, group_col="mchID", cols=None, bins=20, decimals=6):
#     import matplotlib.pyplot as plt

#     if cols is None:
#         cols = [c for c in df.columns if c != group_col]

#     # apply consistent rounding
#     df_rounded = df.copy()
#     df_rounded[cols] = df_rounded[cols].round(decimals)

#     nunique = df_rounded.groupby(group_col)[cols].nunique(dropna=False)

#     for col in cols:
#         values = nunique[col]

#         plt.figure()
#         plt.hist(values, bins=bins)
#         plt.title(f"{col} — nunique per {group_col} (rounded to {decimals})")
#         plt.xlabel("nunique within group")
#         plt.ylabel("count of groups")
#         plt.grid(True)

#         print(f"{col}: min={values.min()}, max={values.max()}, "
#               f"counts={values.value_counts().sort_index().to_dict()}")

#         plt.show()

In [ ]:
# plot_nunique_histograms(df, group_col="mchID", cols=FEATURES, bins=5)

## 16. ONNX

In [ ]:
import sklearn

In [ ]:
model_original = sklearn.base.clone(model)

In [ ]:

model.get_booster().feature_names = [f"f{i}" for i in range(len(FEATURES))]

In [ ]:
import onnx
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort



In [ ]:
print(xgb.__version__)
print(onnx.__version__)

In [ ]:
initial_types = [('float_input', FloatTensorType([-1, len(FEATURES)]))]
onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_types)
onnxmltools.utils.save_model(onnx_model, 'model_placeholder.onnx')

In [ ]:
X_test_numpy = df[FEATURES].to_numpy(dtype=np.float32)

In [ ]:
selectedinput = np.array([[
    5.282344
,   -3.9980261
,   0.9145084
,   -24.896418
,   1.5657754
,   0.29101562
,   94.99072
,   18.571564
,   18.073513
,   18.158918
,   0.07550935
,   46.82407
,   0.185986
,   2.798654
,   3.1521912
,   0.8196365
,   -19.970703
,   14.083008
,   1.3359375
,   2.7854223e-05
,   2.3339007e-05
,   4.8247442e-05
,   0.0043173116
,   10.145764
,   2.4836898
,   -7.150217
,   0.09487186
,   0.56765366
]], dtype=np.float32)
expectedscore = 0.62250906

In [ ]:
# xgb_pred = model.predict_proba(X_test_numpy)[:, 1]
# print(f"XGBoost prediction: {xgb_pred}") 

In [ ]:
# sess = ort.InferenceSession("model.onnx")
# input_name = sess.get_inputs()[0].name
# onnx_label, onnx_prob = sess.run(None, {input_name: X_test_numpy})
# print(onnx_prob)

In [ ]:
# for o in sess.get_outputs():
#     print(o.name, o.shape, o.type)

In [ ]:
# print(sess.get_inputs()[0])

In [ ]:
# diff = np.abs(xgb_pred - onnx_prob[:, 1])  # Assuming onnx_pred is a list of outputs and the probabilities are in the first output
# print("Max abs diff:", np.max(diff))
# print("Mean abs diff:", np.mean(diff))